In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc gem-suite matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

*Estimasi penggunaan: 3 menit pada prosesor Heron r2 (CATATAN: Ini hanya estimasi. Waktu aktual kamu bisa berbeda.)*

## Latar belakang

Tutorial ini mendemonstrasikan cara merealisasikan transisi fase Nishimori pada prosesor kuantum IBM&reg;. Eksperimen ini awalnya dijelaskan dalam [*Realizing the Nishimori transition across the error threshold for constant-depth quantum circuits*](https://arxiv.org/abs/2309.02863).

Transisi fase Nishimori mengacu pada peralihan antara fase teratur jangka pendek dan jangka panjang dalam model Ising random-bond. Pada komputer kuantum, fase teratur jangka panjang terwujud sebagai keadaan di mana Qubit-qubit saling terjerat di seluruh perangkat. Keadaan yang sangat terjerat ini disiapkan menggunakan protokol *generation of entanglement by measurement* (GEM). Dengan memanfaatkan pengukuran mid-circuit, protokol GEM mampu menjerat Qubit-qubit di seluruh perangkat menggunakan Circuit dengan kedalaman konstan saja. Tutorial ini menggunakan implementasi protokol GEM dari paket perangkat lunak [GEM Suite](https://github.com/qiskit-community/gem-suite).

## Persyaratan

Sebelum memulai tutorial ini, pastikan kamu sudah menginstal hal-hal berikut:

- Qiskit SDK v1.0 atau lebih baru, dengan dukungan [visualisasi](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 atau lebih baru ( `pip install qiskit-ibm-runtime` )
- GEM Suite ( `pip install gem-suite` )

## Setup

In [2]:
import matplotlib.pyplot as plt

from collections import defaultdict

from qiskit_ibm_runtime import QiskitRuntimeService

from qiskit.transpiler import generate_preset_pass_manager

from gem_suite import PlaquetteLattice
from gem_suite.experiments import GemExperiment

## Langkah 1: Petakan input klasik ke masalah kuantum
Protokol GEM bekerja pada prosesor kuantum dengan konektivitas Qubit yang digambarkan oleh sebuah kisi. Prosesor kuantum IBM saat ini menggunakan [heavy hex lattice](https://www.ibm.com/quantum/blog/heavy-hex-lattice). Qubit-qubit pada prosesor dikelompokkan ke dalam *plaquette* berdasarkan sel satuan kisi yang ditempatinya. Karena sebuah Qubit bisa muncul di lebih dari satu sel satuan, plaquette-plaquette tersebut tidak saling lepas. Pada heavy hex lattice, sebuah plaquette berisi 12 Qubit. Plaquette-plaquette itu sendiri juga membentuk kisi, di mana dua plaquette terhubung jika mereka berbagi Qubit. Pada heavy hex lattice, plaquette yang bertetangga berbagi 3 Qubit.

Dalam paket perangkat lunak GEM Suite, kelas dasar untuk mengimplementasikan protokol GEM adalah `PlaquetteLattice`, yang merepresentasikan kisi dari plaquette (yang berbeda dari heavy hex lattice). Sebuah `PlaquetteLattice` bisa diinisialisasi dari coupling map Qubit. Saat ini, hanya coupling map heavy hex yang didukung.

Sel kode berikut menginisialisasi plaquette lattice dari coupling map prosesor kuantum IBM. Plaquette lattice tidak selalu mencakup seluruh hardware. Misalnya, `ibm_torino` memiliki total 133 Qubit, tetapi plaquette lattice terbesar yang muat di perangkat tersebut hanya menggunakan 125 di antaranya, dan terdiri dari total 18 plaquette. Hal serupa juga dapat diamati pada perangkat IBM Quantum&reg; dengan jumlah Qubit yang berbeda.

In [ ]:
# QiskitRuntimeService.save_account(channel="ibm_quantum", token="<YOUR_API_KEYN>", overwrite=True, set_as_default=True)
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
plaquette_lattice = PlaquetteLattice.from_coupling_map(backend.coupling_map)

print(f"Number of qubits in backend: {backend.num_qubits}")
print(
    f"Number of qubits in plaquette lattice: {len(list(plaquette_lattice.qubits()))}"
)
print(f"Number of plaquettes: {len(list(plaquette_lattice.plaquettes()))}")

Number of qubits in backend: 133
Number of qubits in plaquette lattice: 125
Number of plaquettes: 18


You can visualize the plaquette lattice by generating a diagram of its graph representation. In the diagram, the plaquettes are represented by labeled hexagons, and two plaquettes are connected by an edge if they share qubits.

In [7]:
plaquette_lattice.draw_plaquettes()

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/625882a4-faeb-4d96-b441-c989f43c4dea-0.avif" alt="Output of the previous code cell" />

Kamu bisa memvisualisasikan plaquette lattice dengan membuat diagram representasi grafnya. Dalam diagram tersebut, plaquette-plaquette direpresentasikan oleh heksagon berlabel, dan dua plaquette dihubungkan oleh sebuah sisi jika mereka berbagi Qubit.

In [8]:
# Get a list of the plaquettes
plaquettes = list(plaquette_lattice.plaquettes())
# Display information about plaquette 0
plaquettes[0]

PyPlaquette(index=0, qubits=[0, 1, 2, 3, 4, 15, 16, 19, 20, 21, 22, 23], neighbors=[3, 1])

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/625882a4-faeb-4d96-b441-c989f43c4dea-0.avif)

Kamu bisa mengambil informasi tentang plaquette individual, seperti Qubit yang dikandungnya, menggunakan metode `plaquettes`.

In [9]:
plaquette_lattice.draw_qubits()

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/a19d63ce-3572-4081-a008-c1332fbbe303-0.avif" alt="Output of the previous code cell" />

In addition to the qubit labels and the edges indicating which qubits are connected, the diagram contains three additional pieces of information that are relevant to the GEM protocol:
- Each qubit is either shaded (gray) or unshaded. The shaded qubits are "site" qubits that represent the sites of the Ising model, and the unshaded qubits are "bond" qubits used to mediate interactions between the site qubits.
- Each site qubit is labeled either (A) or (B), indicating one of two roles a site qubit can play in the GEM protocol (the roles are explained later).
- Each edge is colored using one of six colors, thus partitioning the edges into six groups. This partitioning determines how two-qubit gates can be parallelized, as well as different scheduling patterns that are likely to incur different amounts of error on a noisy quantum processor. Because edges in a group are disjoint, a layer of two-qubit gates can be applied on those edges simultaneously. In fact, it is possible to partition the six colors into three groups of two colors such that the union of each group of two colors is still disjoint. Therefore, only three layers of two-qubit gates are needed to activate every edge. There are 12 ways to so partition the six colors, and each such partition yields a different 3-layer gate schedule.

Now that you have created a plaquette lattice, the next step is to initialize a `GemExperiment` object, passing both the plaquette lattice and the backend that you intend to run the experiment on. The `GemExperiment` class manages the actual implementation of the GEM protocol, including generating circuits, submitting jobs, and analyzing the data. The following code cell initializes the experiment class while restricting the plaquette lattice to only two of the plaquettes (21 qubits), reducing the size of the experiment to ensure that the noise in the hardware doesn't overwhelm the signal.

In [16]:
gem_exp = GemExperiment(plaquette_lattice.filter([9, 12]), backend=backend)

# visualize the plaquette lattice after filtering
plaquette_lattice.filter([9, 12]).draw_qubits()

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/02357c6e-5c83-4ac0-811d-22602d9f33d5-0.avif" alt="Output of the previous code cell" />

Kamu juga bisa membuat diagram Qubit-qubit yang membentuk plaquette lattice.

In [12]:
circuits = gem_exp.circuits()
print(f"Total number of circuits: {len(circuits)}")

Total number of circuits: 252


![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/a19d63ce-3572-4081-a008-c1332fbbe303-0.avif)

Selain label Qubit dan sisi yang menunjukkan Qubit mana yang terhubung, diagram tersebut berisi tiga informasi tambahan yang relevan dengan protokol GEM:
- Setiap Qubit bisa diarsir (abu-abu) atau tidak diarsir. Qubit yang diarsir adalah Qubit "site" yang merepresentasikan situs model Ising, dan Qubit yang tidak diarsir adalah Qubit "bond" yang digunakan untuk memediasi interaksi antara Qubit site.
- Setiap Qubit site diberi label (A) atau (B), menunjukkan salah satu dari dua peran yang bisa dimainkan Qubit site dalam protokol GEM (peran-peran tersebut dijelaskan nanti).
- Setiap sisi diwarnai menggunakan salah satu dari enam warna, sehingga sisi-sisi dibagi menjadi enam kelompok. Pembagian ini menentukan bagaimana Gate dua-Qubit bisa diparalelkan, serta pola penjadwalan berbeda yang kemungkinan akan menghasilkan jumlah error yang berbeda pada prosesor kuantum yang berisik. Karena sisi-sisi dalam satu kelompok saling lepas, lapisan Gate dua-Qubit bisa diterapkan pada sisi-sisi tersebut secara bersamaan. Bahkan, keenam warna tersebut bisa dibagi menjadi tiga pasang warna sedemikian hingga gabungan setiap pasang warna tetap saling lepas. Oleh karena itu, hanya tiga lapisan Gate dua-Qubit yang dibutuhkan untuk mengaktifkan setiap sisi. Ada 12 cara untuk membagi keenam warna tersebut, dan setiap pembagian menghasilkan jadwal Gate 3-lapisan yang berbeda.

Sekarang setelah kamu membuat plaquette lattice, langkah selanjutnya adalah menginisialisasi objek `GemExperiment`, dengan memberikan plaquette lattice dan Backend yang ingin kamu gunakan untuk menjalankan eksperimen. Kelas `GemExperiment` mengelola implementasi aktual protokol GEM, termasuk pembuatan Circuit, pengiriman job, dan analisis data. Sel kode berikut menginisialisasi kelas eksperimen dengan membatasi plaquette lattice hanya pada dua plaquette (21 Qubit), sehingga mengurangi ukuran eksperimen agar noise pada hardware tidak mengalahkan sinyal.

In [13]:
# Restrict experiment to the first scheduling pattern
gem_exp.set_experiment_options(schedule_idx=0)

# There are less circuits now
circuits = gem_exp.circuits()
print(f"Total number of circuits: {len(circuits)}")

# Print the RZZ angles swept over
print(f"RZZ angles:\n{gem_exp.parameters()}")

Total number of circuits: 21
RZZ angles:
[0.         0.07853982 0.15707963 0.23561945 0.31415927 0.39269908
 0.4712389  0.54977871 0.62831853 0.70685835 0.78539816 0.86393798
 0.9424778  1.02101761 1.09955743 1.17809725 1.25663706 1.33517688
 1.41371669 1.49225651 1.57079633]


![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/02357c6e-5c83-4ac0-811d-22602d9f33d5-0.avif)

Sebuah Circuit protokol GEM dibangun menggunakan langkah-langkah berikut:
1. Siapkan keadaan all-$|+\rangle$ dengan menerapkan Gate Hadamard ke setiap Qubit.
2. Terapkan Gate $R_{ZZ}$ antara setiap pasang Qubit yang terhubung. Ini bisa dilakukan menggunakan 3 lapisan Gate. Setiap Gate $R_{ZZ}$ bekerja pada Qubit site dan Qubit bond. Jika Qubit site berlabel (B), maka sudutnya ditetapkan ke $\frac{\pi}{2}$. Jika Qubit site berlabel (A), maka sudutnya boleh bervariasi, menghasilkan Circuit yang berbeda. Secara default, rentang sudut diatur ke 21 titik berjarak sama antara $0$ dan $\frac{\pi}{2}$, inklusif.
3. Ukur setiap Qubit bond dalam basis Pauli $X$. Karena Qubit diukur dalam basis Pauli $Z$, ini bisa dilakukan dengan menerapkan Gate Hadamard sebelum mengukur Qubit.

Perhatikan bahwa paper yang dikutip dalam pengantar tutorial ini menggunakan konvensi berbeda untuk sudut $R_{ZZ}$, yang berbeda dari konvensi yang digunakan dalam tutorial ini dengan faktor 2.

Pada langkah 3, hanya Qubit bond yang diukur. Untuk memahami keadaan apa yang tersisa pada Qubit site, sangat instruktif untuk mempertimbangkan kasus di mana sudut $R_{ZZ}$ yang diterapkan ke Qubit site (A) pada langkah 2 sama dengan $\frac{\pi}{2}$. Dalam kasus ini, Qubit site tersisa dalam keadaan yang sangat terjerat mirip dengan keadaan GHZ,

$$
\lvert \text{GHZ} \rangle = \lvert 00 \cdots 00 \rangle + \lvert 11 \cdots 11 \rangle.
$$

Karena keacakan dalam hasil pengukuran, keadaan aktual dari Qubit site mungkin berupa keadaan berbeda dengan ketertiban jangka panjang, misalnya, $\lvert 00110 \rangle + \lvert 11001 \rangle$. Namun, keadaan GHZ bisa dipulihkan dengan menerapkan operasi dekoding berdasarkan hasil pengukuran. Ketika sudut $R_{ZZ}$ diturunkan dari $\frac{\pi}{2}$, ketertiban jangka panjang masih bisa dipulihkan hingga sudut kritis tertentu, yang tanpa adanya noise, kira-kira sebesar $0.3 \pi$. Di bawah sudut ini, keadaan yang dihasilkan tidak lagi menunjukkan jerat jangka panjang. Peralihan antara ada dan tidak adanya ketertiban jangka panjang inilah yang disebut transisi fase Nishimori.

Dalam deskripsi di atas, Qubit site dibiarkan tidak diukur, dan operasi dekoding bisa dilakukan dengan menerapkan Gate kuantum. Dalam eksperimen yang diimplementasikan dalam GEM Suite, yang diikuti tutorial ini, Qubit site sebenarnya diukur, dan operasi dekoding diterapkan dalam langkah pasca-pemrosesan klasik.

Dalam deskripsi di atas, operasi dekoding bisa dilakukan dengan menerapkan Gate kuantum ke Qubit site untuk memulihkan keadaan kuantum. Namun, jika tujuannya adalah langsung mengukur keadaan tersebut, misalnya untuk keperluan karakterisasi, maka Qubit site diukur bersama dengan Qubit bond, dan operasi dekoding bisa diterapkan dalam langkah pasca-pemrosesan klasik. Inilah cara eksperimen diimplementasikan dalam GEM Suite, yang diikuti tutorial ini.

Selain bergantung pada sudut $R_{ZZ}$ pada langkah 2, yang secara default menyapu 21 nilai, Circuit protokol GEM juga bergantung pada pola penjadwalan yang digunakan untuk mengimplementasikan 3 lapisan Gate $R_{ZZ}$. Seperti yang telah dibahas sebelumnya, ada 12 pola penjadwalan seperti itu. Oleh karena itu, jumlah total Circuit dalam eksperimen adalah $21 \times 12 = 252$.

Circuit-circuit eksperimen bisa dibuat menggunakan metode `circuits` dari kelas `GemExperiment`.

In [14]:
# Get the circuit at index 5
circuit = circuits[5]
# Remove the final measurements to ease visualization
circuit.remove_final_measurements()
# Draw the circuit
circuit.draw("mpl", fold=-1, scale=0.5)

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/fd57d483-c70b-4ad5-b309-15750ad38bac-0.avif" alt="Output of the previous code cell" />

## Step 2: Optimize problem for quantum hardware execution

Transpiling quantum circuits for execution on hardware typically involves a [number of stages](/docs/guides/transpiler-stages). Typically, the stages that incur the most computational overhead are choosing the qubit layout, routing the two-qubit gates to conform to the qubit connectivity of the hardware, and optimizing the circuit to minimize its gate count and depth. In the GEM protocol, the layout and routing stages are unnecessary because the hardware connectivity is already incorporated into the design of the protocol. The circuits already have a qubit layout, and the two-qubit gates are already mapped onto native connections. Furthermore, in order to preserve the structure of the circuit as the $R_{ZZ}$ angle is varied, only very basic circuit optimization should be performed.

The `GemExperiment` class transparently transpiles circuits when executing the experiment. The layout and routing stages are already overridden by default to do nothing, and circuit optimization is performed at a level that only optimizes single-qubit gates. However, you can override or pass additional options using the `set_transpile_options` method. For the sake of visualization, the following code cell manually transpiles the circuit displayed previously, and draws the transpiled circuit.

In [15]:
# Demonstrate setting transpile options
gem_exp.set_transpile_options(
    optimization_level=1  # This is the default optimization level
)
pass_manager = generate_preset_pass_manager(
    backend=backend,
    initial_layout=list(gem_exp.physical_qubits),
    **dict(gem_exp.transpile_options),
)
transpiled = pass_manager.run(circuit)
transpiled.draw("mpl", idle_wires=False, fold=-1, scale=0.5)

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/e9b99d48-8d33-46b5-bff5-480ab1c1c1f2-0.avif" alt="Output of the previous code cell" />

Untuk keperluan tutorial ini, cukup mempertimbangkan satu pola penjadwalan saja. Sel kode berikut membatasi eksperimen pada pola penjadwalan pertama. Hasilnya, eksperimen hanya memiliki 21 Circuit, satu untuk setiap sudut $R_{ZZ}$ yang disapu.

In [10]:
exp_data = gem_exp.run(shots=10_000)

To wait for the results, call the `block_for_results` method of the `ExperimentData` object. This call will cause the interpreter to hang until the jobs are finished.

In [11]:
exp_data.block_for_results()

ExperimentData(GemExperiment, d0d5880a-34c1-4aab-a7b6-c4f58516bc03, job_ids=['cwg12ptmptp00082khhg'], metadata=<5 items>, figure_names=['two_point_correlation.svg', 'normalized_variance.svg', 'plaquette_ops.svg', 'bond_ops.svg'])

Sel kode berikut menggambar diagram Circuit pada indeks 5. Untuk mengurangi ukuran diagram, Gate pengukuran di akhir Circuit dihapus.

In [ ]:
def magnetization_distribution(
    counts_dict: dict[str, int],
) -> dict[str, float]:
    """Compute magnetization distribution from counts dictionary."""
    # Construct dictionary from magnetization to count
    mag_dist = defaultdict(float)
    for bitstring, count in counts_dict.items():
        mag = bitstring.count("0") - bitstring.count("1")
        mag_dist[mag] += count
    # Normalize
    shots = sum(counts_dict.values())
    for mag in mag_dist:
        mag_dist[mag] /= shots
    return mag_dist


# Get counts dictionaries with and without decoding
data = exp_data.data()
# Get the last data point, which is at the angle for the GHZ state
raw_counts = data[-1]["counts"]
# Without decoding
site_indices = [
    i for i, q in enumerate(gem_exp.plaquettes.qubits()) if q.role == "Site"
]
site_raw_counts = defaultdict(int)
for key, val in raw_counts.items():
    site_str = "".join(key[-1 - i] for i in site_indices)
    site_raw_counts[site_str] += val
# With decoding
_, site_decoded_counts = gem_exp.plaquettes.decode_outcomes(
    raw_counts, return_counts=True
)

# Compute magnetization distribution
raw_magnetization = magnetization_distribution(site_raw_counts)
decoded_magnetization = magnetization_distribution(site_decoded_counts)

# Plot
plt.bar(*zip(*raw_magnetization.items()), label="raw")
plt.bar(*zip(*decoded_magnetization.items()), label="decoded", width=0.3)
plt.legend()
plt.xlabel("Magnetization")
plt.ylabel("Frequency")
plt.title("Magnetization distribution with and without decoding")

Text(0.5, 1.0, 'Magnetization distribution with and without decoding')

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/8ead3582-16df-4616-836c-bdce867ad6b8-1.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/fd57d483-c70b-4ad5-b309-15750ad38bac-0.avif)

## Langkah 2: Optimalkan masalah untuk eksekusi hardware kuantum
Transpilasi Circuit kuantum untuk eksekusi pada hardware biasanya melibatkan [sejumlah tahap](/guides/transpiler-stages). Umumnya, tahap yang paling banyak membutuhkan overhead komputasi adalah memilih tata letak Qubit, merutekan Gate dua-Qubit agar sesuai dengan konektivitas Qubit hardware, dan mengoptimalkan Circuit untuk meminimalkan jumlah Gate dan kedalamannya. Dalam protokol GEM, tahap tata letak dan perutean tidak diperlukan karena konektivitas hardware sudah dimasukkan dalam desain protokol. Circuit sudah memiliki tata letak Qubit, dan Gate dua-Qubit sudah dipetakan ke koneksi native. Selain itu, untuk menjaga struktur Circuit saat sudut $R_{ZZ}$ divariasikan, hanya optimisasi Circuit yang sangat dasar yang boleh dilakukan.

Kelas `GemExperiment` secara transparan melakukan transpilasi Circuit saat menjalankan eksperimen. Tahap tata letak dan perutean sudah dioverride secara default agar tidak melakukan apa-apa, dan optimisasi Circuit dilakukan pada tingkat yang hanya mengoptimalkan Gate single-Qubit. Namun, kamu bisa mengoverride atau memberikan opsi tambahan menggunakan metode `set_transpile_options`. Untuk keperluan visualisasi, sel kode berikut secara manual mentranspilasi Circuit yang ditampilkan sebelumnya, dan menggambar Circuit yang sudah ditranspilasi.

In [13]:
exp_data.figure("two_point_correlation")

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/4ecb25c8-e572-49af-a879-9943039db131-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/e9b99d48-8d33-46b5-bff5-480ab1c1c1f2-0.avif)

## Langkah 3: Eksekusi menggunakan primitif Qiskit
Untuk mengeksekusi Circuit protokol GEM pada hardware, panggil metode `run` dari objek `GemExperiment`. Kamu bisa menentukan jumlah shot yang ingin kamu sampel dari setiap Circuit. Metode `run` mengembalikan objek [ExperimentData](https://qiskit-community.github.io/qiskit-experiments/stubs/qiskit_experiments.framework.ExperimentData.html) yang harus kamu simpan ke variabel. Perhatikan bahwa metode `run` hanya mengirimkan job tanpa menunggu selesai, jadi ini adalah panggilan non-blocking.

In [14]:
exp_data.figure("normalized_variance")

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/2b351d68-3924-445a-94ef-047b16214e8a-0.avif" alt="Output of the previous code cell" />

Untuk menunggu hasilnya, panggil metode `block_for_results` dari objek `ExperimentData`. Panggilan ini akan membuat interpreter berhenti sampai job selesai.

In [15]:
gem_exp = GemExperiment(
    plaquette_lattice.filter(range(3, 9)), backend=backend
)
gem_exp.set_experiment_options(schedule_idx=0)
exp_data = gem_exp.run(shots=10_000)
exp_data.block_for_results()
exp_data.figure("normalized_variance")

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/08581c09-a6a5-4a56-9fc4-abf22b063c6a-0.avif" alt="Output of the previous code cell" />

In [16]:
gem_exp = GemExperiment(plaquette_lattice, backend=backend)
gem_exp.set_experiment_options(schedule_idx=0)
exp_data = gem_exp.run(shots=10_000)
exp_data.block_for_results()
exp_data.figure("normalized_variance")

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/37e9a4cd-6efb-4ade-ad09-8139db9d58e9-0.avif" alt="Output of the previous code cell" />

## Langkah 4: Pasca-proses dan kembalikan hasil dalam format klasik yang diinginkan
Pada sudut $R_{ZZ}$ sebesar $\frac{\pi}{2}$, keadaan yang terdekode akan menjadi keadaan GHZ tanpa adanya noise. Ketertiban jangka panjang dari keadaan GHZ bisa divisualisasikan dengan memplot magnetisasi dari bitstring yang diukur. Magnetisasi $M$ didefinisikan sebagai jumlah dari operator Pauli $Z$ single-Qubit,
$$
M = \sum_{j=1}^N Z_j,
$$
di mana $N$ adalah jumlah Qubit site. Nilainya untuk sebuah bitstring sama dengan selisih antara jumlah nol dan jumlah satu. Mengukur keadaan GHZ menghasilkan keadaan all-zeros atau all-ones dengan probabilitas yang sama, sehingga magnetisasinya akan bernilai $+N$ setengah waktu dan $-N$ setengah waktu lainnya. Jika ada error akibat noise, nilai-nilai lain juga akan muncul, tetapi jika noise tidak terlalu besar, distribusinya masih akan memuncak di dekat $+N$ dan $-N$.

Untuk bitstring mentah sebelum dekoding, distribusi magnetisasi akan setara dengan bitstring yang seragam acak, tanpa adanya noise.

Sel kode berikut memplot magnetisasi dari bitstring mentah dan bitstring yang terdekode pada sudut $R_{ZZ}$ sebesar $\frac{\pi}{2}$.